# Signal Filtering 3 — Frequency-Domain & Shape (Butterworth / Savitzky-Golay)
**Certificate 04 · Data Analytics**  |  [← Back to the Lab Hub](../../index.ipynb)

A noisy **weight / load-cell** signal: fill/rest/empty cycles with quantization dither, agitation/sloshing, and washing-fluid spikes. numpy, scipy, pandas and matplotlib are pre-installed in Colab.

> Runs on a synthetic signal that mimics a real load cell. **To use your own export**, replace the load cell with `pd.read_csv("your_export.csv", ...)` (two columns: timestamp, value).

## Objectives
By the end you will be able to:
- Resample an irregular signal to a uniform grid (required for these filters).
- Apply a zero-phase Butterworth low-pass and choose its cutoff.
- Use Savitzky-Golay to preserve fill-edge shape, and a median-then-Butterworth composite for spikes.

## Load & make a uniform grid
Butterworth and Savitzky-Golay need evenly-spaced samples, so first resample the irregular signal to 1 s and interpolate.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
DATA = "https://raw.githubusercontent.com/IF-JL/COEFAM-Labs/main/labs/cert04_data_analytics/data/"
s = pd.read_csv(DATA + "weight_noisy.csv", parse_dates=["timestamp"]).set_index("timestamp")["weight"]
print("samples:", s.size, "| range: %.1f .. %.1f" % (s.min(), s.max()))
s.head()

In [ ]:
u = s.resample("1s").mean().interpolate("linear")
print("uniform samples:", u.size)

## 1 · Zero-phase Butterworth low-pass
`filtfilt` runs the filter forward then backward, so there is **no lag** (a plain causal filter would shift the signal in time).

In [ ]:
from scipy.signal import butter, filtfilt

def butter_lp(x, order=4, cutoff_hz=0.05, fs=1.0):
    nyq = fs / 2.0
    wn = min(cutoff_hz, 0.99 * nyq) / nyq
    b, a = butter(order, wn, btype="low")
    return pd.Series(filtfilt(b, a, x.to_numpy(float)), index=x.index)

lo, hi = 3000, 3600
w = u.iloc[lo:hi]
plt.figure(figsize=(11,3))
plt.plot(w.index, w.values, lw=.5, alpha=.5, label="raw")
plt.plot(w.index, butter_lp(u, 4, 0.05).iloc[lo:hi].values, label="Butterworth low-pass (0.05 Hz)")
plt.legend(); plt.title("Zero-phase Butterworth (filtfilt - no lag)"); plt.show()

## 2 · Choosing the cutoff
Lower cutoff = smoother but slower to follow real changes. Pick it just above where the real fills live.

In [ ]:
plt.figure(figsize=(11,3))
plt.plot(w.index, w.values, lw=.4, alpha=.3, label="raw")
for c in [0.02, 0.05, 0.1]:
    plt.plot(w.index, butter_lp(u, 4, c).iloc[lo:hi].values, label=f"cutoff {c} Hz")
plt.legend(); plt.title("Butterworth cutoff trade-off"); plt.show()

## 3 · Savitzky-Golay preserves shape
A big moving average rounds off the fill ramp; Savitzky-Golay fits a local polynomial, keeping the edge.

In [ ]:
from scipy.signal import savgol_filter

def savgol(x, wl=51, po=3):
    wl = wl if wl % 2 else wl + 1
    return pd.Series(savgol_filter(x.to_numpy(float), wl, po), index=x.index)

a, b = 1900, 2600   # around a fill transition
seg = u.iloc[a:b]
plt.figure(figsize=(11,3))
plt.plot(seg.index, seg.values, lw=.5, alpha=.4, label="raw")
plt.plot(seg.index, savgol(u, 51, 3).iloc[a:b].values, label="Savitzky-Golay (51, 3)")
plt.plot(seg.index, u.rolling(51, center=True, min_periods=1).mean().iloc[a:b].values, label="moving average (51) - rounds the edge")
plt.legend(); plt.title("Savitzky-Golay preserves the fill-edge shape"); plt.show()

## 4 · Composite: median -> Butterworth
The load-cell engineer's default: a short median kills the spikes, then Butterworth smooths the rest.

In [ ]:
from scipy.signal import medfilt

def median_then_butter(x, kernel=5, order=4, cutoff=0.05, fs=1.0):
    m = pd.Series(medfilt(x.to_numpy(float), kernel), index=x.index)
    return butter_lp(m, order, cutoff, fs)

c, d = 500, 1100    # window with spikes
w2 = u.iloc[c:d]
plt.figure(figsize=(11,3))
plt.plot(w2.index, w2.values, lw=.5, alpha=.4, label="raw (spikes)")
plt.plot(w2.index, median_then_butter(u).iloc[c:d].values, label="median -> Butterworth")
plt.legend(); plt.title("Composite: median rejects spikes, Butterworth smooths"); plt.show()

## Debrief
- **filtfilt** gives zero lag; the **cutoff** is chosen from the noise spectrum (Filtering 1); **Savitzky-Golay** protects real transitions; the **composite** handles spikes + noise together.
- These are the strategies in the internal weight-filter benchmark - a natural next lab is to *score* them all and pick a winner (noise reduction vs lag vs smoothness).